# Code Generator

Using a Frontier model to generate high performance C++ code from Python code.

In [1]:
# Imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [2]:
# Connecting to OpenAI API:
load_dotenv(override= True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print('OPENAI API KEY FOUND!')
else:
    print('OPENAI API KEY NOT FOUND!')

OPENAI_MODEL = 'gpt-5'

# Want to keep costs ultra-low? Uncomment this line:
#OPENAI_MODEL = 'gpt-5-nano'

openai = OpenAI()


OPENAI API KEY FOUND!


## PLEASE NOTE:

We will be writing a solution to convert Python into efficient, optimized C++ code for your machine, which can be compiled to native machine code and executed.

It is not necessary for you to execute the code yourself - that's not the point of the exercise!

But if you would like to (because it's satisfying!) then I'm including the steps here. Very optional!

As an alternative, I'll also show you a website where you can run the C++ code.

In [3]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '13th Gen Intel(R) Core(TM) i9-13900HX',
  'cores_logical': 32,
  'cores_physical': 24,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

# Prompt to get steps to be able to run C++ code:
Run the below prompt and generate response to get instructions about how to run C++ code natively:

message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL,
                                          messages= [{'role': 'user', 'content': message}])

display(Markdown(response.choices[0].message.content))

## Note:

The first time I ran the above cell, I got these instructions to install environment and compiler to run C++ code:

```
Short answer: you don’t currently have a C++ compiler installed. You’ll need to install one.

Simplest way on your system (Windows 10 with winget): install MSYS2 and its GCC (MinGW-w64) toolchain, then compile with g++.

Steps

Install MSYS2 via winget (default path is C:\msys64)
Open PowerShell (Run as Administrator) and run: winget install -e --id MSYS2.MSYS2
Install the 64-bit GCC toolchain (UCRT variant)
Run this to install the compiler: C:\msys64\usr\bin\bash -lc "pacman -Sy --noconfirm && pacman -S --noconfirm --needed mingw-w64-ucrt-x86_64-gcc"
Verify it works: C:\msys64\ucrt64\bin\g++.exe --version
You do not need to change PATH; we’ll call g++ with its full path from Python.

Python commands (fast runtime focus)

This compiles main.cpp with high optimization, CPU-specific tuning, and link-time optimization. It outputs main.exe in the current directory.
compile_command = [ r"C:\msys64\ucrt64\bin\g++.exe", "-std=c++20", "-O3", "-march=native", "-DNDEBUG", "-flto", "main.cpp", "-o", "main.exe", "-static-libstdc++", "-static-libgcc", ]

run_command = ["main.exe"]

Notes

Place main.cpp in your current working directory (where the Python script runs).
If you installed MSYS2 somewhere other than C:\msys64, adjust the g++.exe path accordingly.
-march=native generates code optimized for your CPU (13th Gen Intel), which gives best performance on this machine. If you need portability to other PCs, remove -march=native.
```

So, I followed these steps to follow the instructions given:

1. Open PowerShell as Administrator

- Click your Windows Start Menu.
- Type PowerShell.
- Right-click on Windows PowerShell in the search results and select Run as administrator. (Click "Yes" if Windows asks for permission).

2. Install the MSYS2 Base Environment

- Copy this exact command, paste it into your PowerShell window, and press Enter:
```winget install -e --id MSYS2.MSYS2```

- What this is doing: winget is downloading a program called MSYS2. Think of MSYS2 as a tiny, lightweight Linux-like environment that runs inside Windows. C++ compilers love Linux environments, so this acts as a bridge.


3. Install the Actual GCC Compiler inside MSYS2

- Wait for Step 2 to completely finish. Once you see the PowerShell prompt again, copy and paste this entire line and press Enter:
```C:\msys64\usr\bin\bash -lc "pacman -Sy --noconfirm && pacman -S --noconfirm --needed mingw-w64-ucrt-x86_64-gcc"```
- What this is doing: This looks intimidating, but it's just telling the MSYS2 environment you just installed to open up, use its own internal downloader tool (called pacman), and download the gcc compiler.

4. Verify it Worked

Once the download from Step 3 finishes, paste this to check if the compiler is alive:

```C:\msys64\ucrt64\bin\g++.exe --version```
- If this prints out a few lines of text starting with g++.exe (RevX, Built by MSYS2 project)..., you have successfully installed the compiler! You can now close PowerShell.

In [4]:
# Compile and Run Commands for GCC (C++ Compiler) (after following the above instructions):

compile_command = [ r"C:\msys64\ucrt64\bin\g++.exe", "-std=c++20", "-O3", "-march=native", "-DNDEBUG", "-flto", "main.cpp", "-o", "main.exe", "-static-libstdc++", "-static-libgcc", ]

run_command = ["main.exe"]

## Code Porting:

In [6]:
# System Prompt:
system_prompt = '''
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
'''

# Function to create user prompt with given Python code:
def user_prompt_for(python):
    return f'''
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
'''